In [0]:
from pyspark.sql import functions as F


# ----- Customers (static sample) -----
customers = [
  (101, "Asha",   "Chennai",   "Retail"),
  (102, "Bala",   "Bengaluru", "SMB"),
  (103, "Charan", "Hyderabad", "Enterprise")
]

cust_df = spark.createDataFrame(customers, ["customer_id","customer_name","city","segment"])

(cust_df.write.format("delta")
 .mode("overwrite")
 .saveAsTable("project.bronze.customers"))

# ----- Orders (sample: some for today, some for yesterday) -----
today = F.current_date()
yesterday = F.date_sub(F.current_date(), 1)

orders = [
  (1, 101, "NEW",       "2025-12-15 10:00:00", "today"),
  (2, 102, "SHIPPED",   "2025-12-15 12:15:00", "today"),
  (3, 103, "CANCELLED", "2025-12-14 09:30:00", "yesterday")
]

orders_df = (spark.createDataFrame(orders, ["order_id","customer_id","status","order_ts","tag"])
             .withColumn("order_ts", F.to_timestamp("order_ts"))
             .withColumn("ingest_date", F.when(F.col("tag")=="today", today).otherwise(yesterday))
             .drop("tag"))

(orders_df.write.format("delta")
 .mode("overwrite")
 .saveAsTable("project.bronze.orders"))





In [0]:
%sql
select * from project.bronze.orders;

order_id,customer_id,status,order_ts,ingest_date
1,101,NEW,2025-12-15T10:00:00Z,2025-12-15
2,102,SHIPPED,2025-12-15T12:15:00Z,2025-12-15
3,103,CANCELLED,2025-12-14T09:30:00Z,2025-12-14


In [0]:
%sql
select * from project.bronze.customers;

customer_id,customer_name,city,segment
101,Asha,Chennai,Retail
102,Bala,Bengaluru,SMB
103,Charan,Hyderabad,Enterprise
